In [ ]:
!pip install torch torchvision
!pip install opencv-python-headless


  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from scipy.ndimage import label
import json
from scipy.spatial.distance import cdist

# Load the pre-trained model
model = torch.hub.load('pytorch/vision:v0.10.0', 'deeplabv3_resnet50', pretrained=True)
model.eval()

# Pre-defined mask for images without annotation files
def get_predefined_mask(image_shape):
    mask = np.zeros(image_shape, dtype=np.uint8)
    # Extract the height and width of the image
    height, width = image_shape
    # Define the region you consider as the lane
    lane_width = width // 2
    # Draw a rectangle from the center of the image to the bottom
    cv2.rectangle(mask, (lane_width // 2, height // 2), (lane_width + lane_width // 2, height), 255, thickness=-1)
    return mask.astype(bool)


# Function to preprocess the input image
def preprocess_image(image_path):
    input_image = Image.open(image_path).convert("RGB")
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_tensor = preprocess(input_image)
    input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch as expected by the model
    return input_batch, input_image

# Function to perform inference
def perform_inference(input_batch):
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        model.to('cuda')

    with torch.no_grad():
        output = model(input_batch)['out'][0]
    return output

# Function to decode the segmentation map
def decode_segmap(image, nc=21):
    label_colors = np.array([
        (0, 0, 0),       # 0=background
        (128, 0, 0),     # 1=aeroplane
        (0, 128, 0),     # 2=bicycle
        (128, 128, 0),   # 3=bird
        (0, 0, 128),     # 4=boat
        (128, 0, 128),   # 5=bottle
        (0, 128, 128),   # 6=bus
        (128, 128, 128), # 7=car
        (64, 0, 0),      # 8=cat
        (192, 0, 0),     # 9=chair
        (64, 128, 0),    # 10=cow
        (192, 128, 0),   # 11=diningtable
        (64, 0, 128),    # 12=dog
        (192, 0, 128),   # 13=horse
        (64, 128, 128),  # 14=motorbike
        (192, 128, 128), # 15=person
        (0, 64, 0),      # 16=potted plant
        (128, 64, 0),    # 17=sheep
        (0, 192, 0),     # 18=sofa
        (128, 192, 0),   # 19=train
        (0, 64, 128)     # 20=tv/monitor
    ])
    r = np.zeros_like(image).astype(np.uint8)
    g = np.zeros_like(image).astype(np.uint8)
    b = np.zeros_like(image).astype(np.uint8)
    for l in range(0, nc):
        idx = image == l
        r[idx] = label_colors[l, 0]
        g[idx] = label_colors[l, 1]
        b[idx] = label_colors[l, 2]
    rgb = np.stack([r, g, b], axis=2)
    return rgb

# Function to create a mask from polygons
def create_mask_from_polygons(image_shape, polygons):
    mask = np.zeros(image_shape, dtype=np.uint8)
    for polygon in polygons:
        try:
            coords = np.array(polygon['geometry']['coordinates'][0], dtype=np.int32)
            if coords.shape[0] >= 3:  # Ensure the polygon has at least 3 points
                cv2.fillPoly(mask, [coords], 1)
            else:
                print(f"Skipping polygon with insufficient points: {coords}")
        except Exception as e:
            print(f"Error processing polygon: {polygon}")
            print(f"Exception: {e}")
    return mask.astype(bool)

# Function to load ego road annotations
def load_ego_road_annotations(ego_road_annotation_path):
    with open(ego_road_annotation_path, 'r') as f:
        ego_road_data = json.load(f)
    return ego_road_data

# Function to filter vehicles on the ego road
def filter_vehicles_on_ego_road(output_predictions, mask):
    vehicle_classes = [7, 6]
    vehicle_mask = np.isin(output_predictions.cpu().numpy(), vehicle_classes)
    ego_road_vehicles_mask = vehicle_mask & mask
    # Use connected component analysis to count distinct vehicles
    labeled_array, num_features = label(ego_road_vehicles_mask)
    return ego_road_vehicles_mask, num_features, labeled_array

# Function to filter pedestrians on the ego road
def filter_pedestrians_on_ego_road(output_predictions, mask):
    pedestrian_class = 15
    pedestrian_mask = output_predictions.cpu().numpy() == pedestrian_class
    ego_road_pedestrians_mask = pedestrian_mask & mask
    # Use connected component analysis to count distinct pedestrians
    labeled_array, num_features = label(ego_road_pedestrians_mask)
    return ego_road_pedestrians_mask, num_features, labeled_array

# Function to calculate the distance to the closest vehicle
def calculate_closest_vehicle_distance(labeled_array, input_image):
    vehicle_coords = np.argwhere(labeled_array > 0)
    ego_vehicle_position = np.array([input_image.size[1] // 2, input_image.size[0]])
    if vehicle_coords.size == 0:
        return float('inf'), None
    distances = cdist(vehicle_coords, [ego_vehicle_position])
    closest_idx = np.argmin(distances)
    closest_vehicle_distance = distances[closest_idx][0]
    closest_vehicle_coords = vehicle_coords[closest_idx]
    return closest_vehicle_distance, closest_vehicle_coords

# Function to display the image
def display_image(image, title):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

import os

# Main function to run the entire process
def main(image_path, ego_road_annotation_path):
    input_batch, input_image = preprocess_image(image_path)
    output = perform_inference(input_batch)
    output_predictions = output.argmax(0)

    display_image(input_image, "Input Image")

    segmented_image = decode_segmap(output_predictions.cpu().numpy())
    display_image(segmented_image, "Segmentation Map")

    if os.path.exists(ego_road_annotation_path):
        ego_road_annotations = load_ego_road_annotations(ego_road_annotation_path)
        mask = create_mask_from_polygons(segmented_image.shape[:2], ego_road_annotations)
    else:
        mask = get_predefined_mask(segmented_image.shape[:2])

    # Visualize the ROI on the original image
    roi_image = np.array(input_image)
    roi_color = [255, 0, 0]  # Red color for ROI
    roi_image[mask] = roi_color
    display_image(roi_image, "Region of Interest (ROI)")

    ego_road_vehicles_mask, vehicle_count, labeled_array = filter_vehicles_on_ego_road(output_predictions, mask)

    # Visualize the filtered vehicles on the original image
    vehicles_image = np.array(input_image)
    vehicles_image[ego_road_vehicles_mask] = [0, 255, 0]  # Green color for detected vehicles
    display_image(vehicles_image, f"Vehicles on Ego Road (Count: {vehicle_count})")

    print(f"Number of vehicles detected on ego road: {vehicle_count}")

    # Detect pedestrians on ego road and surrounding area
    ego_road_pedestrians_mask, pedestrian_count, _ = filter_pedestrians_on_ego_road(output_predictions, mask)

    # Visualize the filtered pedestrians on the original image
    pedestrians_image = np.array(input_image)
    pedestrians_image[ego_road_pedestrians_mask] = [0, 0, 255]  # Blue color for detected pedestrians
    display_image(pedestrians_image, f"Pedestrians on Ego Road (Count: {pedestrian_count})")

    print(f"Number of pedestrians detected on ego road: {pedestrian_count}")

    # Calculate and visualize the closest vehicle distance
    closest_vehicle_distance, closest_vehicle_coords = calculate_closest_vehicle_distance(labeled_array, input_image)
    print(f"Distance to the closest vehicle: {closest_vehicle_distance} pixels")

    if closest_vehicle_coords is not None:
        cv2.circle(vehicles_image, tuple(closest_vehicle_coords[::-1]), 10, (255, 0, 0), -1)  # Red circle for closest vehicle
        display_image(vehicles_image, f"Closest Vehicle (Distance: {closest_vehicle_distance} pixels)")

# Run the main function
image_path = "/content/drive/MyDrive/Colab Notebooks/ZODMiniDataset/Frames/single_frames/018591/camera_front_dnat/018591_india_2020-07-24T07:08:56.778861Z.jpg"
ego_road_annotation_path = "/content/drive/MyDrive/Colab Notebooks/ZODMiniDataset/Frames/single_frames/018591/annotations/ego_road.json"
main(image_path, ego_road_annotation_path)

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Load DeepLabV3 model for segmentation
deeplab_model = torch.hub.load('pytorch/vision:v0.10.0', 'deeplabv3_resnet101', pretrained=True)
deeplab_model.eval()

# Load YOLOv5 model for object detection
yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Function to preprocess the input image
def preprocess_image(image_path):
    input_image = Image.open(image_path).convert("RGB")
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_tensor = preprocess(input_image)
    input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch as expected by the model
    return input_batch, input_image

# Perform inference using DeepLabV3
def perform_inference(input_batch):
    with torch.no_grad():
        output = deeplab_model(input_batch)['out'][0]
    return output

# Decode the segmentation map
def decode_segmap(image, nc=21):
    label_colors = np.array([(0, 0, 0),  # 0=background
                             (128, 64, 128),  # 1=road
                             (70, 70, 70), (102, 102, 156), (190, 153, 153), (153, 153, 153),
                             (250, 170, 30), (220, 220, 0), (107, 142, 35), (152, 251, 152),
                             (70, 130, 180), (220, 20, 60), (255, 0, 0), (0, 0, 142),
                             (0, 0, 70), (0, 60, 100), (0, 80, 100), (0, 0, 230),
                             (119, 11, 32), (128, 192, 0), (255, 128, 0)])  # Add colors for each class here
    r = np.zeros_like(image).astype(np.uint8)
    g = np.zeros_like(image).astype(np.uint8)
    b = np.zeros_like(image).astype(np.uint8)
    for l in range(0, nc):
        idx = image == l
        r[idx] = label_colors[l, 0]
        g[idx] = label_colors[l, 1]
        b[idx] = label_colors[l, 2]
    rgb = np.stack([r, g, b], axis=2)
    return rgb

# Function to display images
def display_image(image, title="Image"):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

# Function to load ego road annotations (for example, from a JSON file)
def load_ego_road_annotations(ego_road_annotation_path):
    with open(ego_road_annotation_path, 'r') as f:
        ego_road_data = json.load(f)
    return ego_road_data

# Function to create a mask from polygons
def create_mask_from_polygons(image_shape, polygons):
    mask = np.zeros(image_shape, dtype=np.uint8)
    for polygon in polygons:
        try:
            coords = np.array(polygon['geometry']['coordinates'][0], dtype=np.int32)
            if coords.shape[0] >= 3:  # Ensure the polygon has at least 3 points
                cv2.fillPoly(mask, [coords], 1)
            else:
                print(f"Skipping polygon with insufficient points: {coords}")
        except Exception as e:
            print(f"Error processing polygon: {polygon}")
            print(f"Exception: {e}")
    return mask.astype(bool)

# Perform object detection using YOLOv5
def detect_objects(image_path):
    results = yolo_model(image_path)
    return results

# Plot YOLOv5 detection results
def plot_results(results, image_path, mask=None):
    img = cv2.imread(image_path)
    for *xyxy, conf, cls in results.xyxy[0]:
        if mask is not None:
            # Check if the center of the bounding box is inside the mask
            center_x = int((xyxy[0] + xyxy[2]) / 2)
            center_y = int((xyxy[1] + xyxy[3]) / 2)
            if not mask[center_y, center_x]:
                continue  # Skip this detection if it's not in the ego road region
        label = f'{results.names[int(cls)]} {conf:.2f}'
        plot_one_box(xyxy, img, label=label, color=colors(int(cls), True), line_thickness=3)
    plt.imshow(img[:, :, ::-1])
    plt.show()

# Function to plot one bounding box
def plot_one_box(x, img, color=None, label=None, line_thickness=None):
    tl = line_thickness or round(0.002 * max(img.shape[0:2]))
    color = color or [random.randint(0, 255) for _ in range(3)]
    c1, c2 = (int(x[0]), int(x[1])), (int(x[2]), int(x[3]))
    cv2.rectangle(img, c1, c2, color, thickness=tl, lineType=cv2.LINE_AA)
    if label:
        tf = max(tl - 1, 1)
        t_size = cv2.getTextSize(label, 0, fontScale=tl / 3, thickness=tf)[0]
        c2 = c1[0] + t_size[0], c1[1] - t_size[1] - 3
        cv2.rectangle(img, c1, c2, color, -1, cv2.LINE_AA)
        cv2.putText(img, label, (c1[0], c1[1] - 2), 0, tl / 3, [225, 255, 255], thickness=tf, lineType=cv2.LINE_AA)

# Main function
def main(image_path, ego_road_annotation_path):
    input_batch, input_image = preprocess_image(image_path)
    output = perform_inference(input_batch)
    output_predictions = output.argmax(0)

    display_image(input_image, "Input Image")

    segmented_image = decode_segmap(output_predictions.cpu().numpy())
    display_image(segmented_image, "Segmentation Map")

    ego_road_annotations = load_ego_road_annotations(ego_road_annotation_path)
    mask = create_mask_from_polygons(segmented_image.shape[:2], ego_road_annotations)

    # Visualize the ROI on the original image
    roi_image = np.array(input_image)
    roi_color = [255, 0, 0]  # Red color for ROI
    roi_image[mask] = roi_color
    display_image(roi_image, "Region of Interest (ROI)")

    # Use YOLOv5 to detect objects
    results = detect_objects(image_path)

    # Plot results and filter detections based on the mask
    plot_results(results, image_path, mask=mask)

# Run the main function
image_path = "/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames/044953/camera_front_blur/044953_india_2020-05-06T11:41:57.779576Z.jpg"
ego_road_annotation_path = "/content/drive/My Drive/Colab Notebooks/ZODMiniDataset/Frames/single_frames/044953/annotations/ego_road.json"
main(image_path, ego_road_annotation_path)


Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-7-21 Python-3.10.12 torch-2.3.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Skipping polygon with insufficient points: [1805 1199]
Skipping polygon with insufficient points: [1946 1172]
Skipping polygon with insufficient points: [1209 1395]
Skipping polygon with insufficient points: [2206 1180]
Skipping polygon with insufficient points: [2130 1161]
